# 10.2 — Exact, Approximate, and Mergeable

**Chapter 10, section 3.1** — the aggregate classification of \Cref{tab:AggregateCost} and the
sample-versus-sketch distinction that follows it. The starting point for **Exercises 5 and 6**.

**The question this notebook answers:** what does an approximate distinct count actually buy, what
does its accuracy parameter actually promise, and what can a sketch do that a sample cannot?

\Cref{tab:AggregateCost} sorts aggregates by what they must remember: *constant* for `count` and
`sum`, *O(distinct values)* for `count_distinct`, and *configurable* for the approximate forms,
where "a fixed-size summary is kept instead of the values, and its size sets the accuracy". Three
claims are measured here — the cost of exactness, the meaning of the accuracy parameter, and
mergeability.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, time, logging, tempfile, statistics
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))

spark = (SparkSession.builder
         .appName("CS777-10.2")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         # An exact count_distinct over 731,884 unique ids keeps all 731,884 of them,
         # which is this notebook's first point and needs more than the default local heap.
         .config("spark.driver.memory", "2g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Section 2 calls approx_count_distinct with an rsd Spark rejects, on purpose. PySpark
# logs a structured record carrying the whole JVM stack trace for every analysis error,
# which would bury the one line that is the lesson.
logging.getLogger("DataFrameQueryContextLogger").setLevel(logging.CRITICAL)

events = (spark.read.parquet(os.path.join(DATA, "gdelt", "gdelt-events-small.parquet"))
          .select("GLOBALEVENTID", "SQLDATE", "Actor1Geo_FullName", "AvgTone")
          .cache())
N = events.count()
heap_mb = spark.sparkContext._jvm.java.lang.Runtime.getRuntime().maxMemory() // 10**6
print(f"rows {N:,}   days {events.select('SQLDATE').distinct().count()}   "
      f"driver heap {heap_mb} MB")

rows 731,884   days 35   driver heap 2147 MB


---

## 1. What exactness costs, and on which column

Two columns, deliberately chosen at different cardinalities:

* **`GLOBALEVENTID`** — the event id, so every row is distinct. The worst case: the set an exact
  count has to retain is as large as the table.
* **`Actor1Geo_FullName`** — a place name, tens of thousands of distinct values in this extract.

`count_distinct` must remember every value it has already seen in order to recognise a repeat.
`approx_count_distinct` keeps a fixed-size sketch instead. At this scale both finish, so what the
comparison shows is the *shape* of the trade rather than a rescue: the exact answer is exact and its
state grows with the column's cardinality, and the approximate answer is wrong by a little and its
state does not.

In [2]:
def timed(col_expr):
    t = time.perf_counter()
    v = events.select(col_expr).first()[0]
    return v, time.perf_counter() - t

print(f"{'column':<22}{'exact':>10}{'s':>7}{'approx':>10}{'s':>7}{'error':>9}")
print("-" * 65)
truth = {}
for col in ["GLOBALEVENTID", "Actor1Geo_FullName"]:
    exact, te = timed(F.count_distinct(col))
    approx, ta = timed(F.approx_count_distinct(col))      # rsd defaults to 0.05
    truth[col] = exact
    print(f"{col:<22}{exact:>10,}{te:>7.2f}{approx:>10,}{ta:>7.2f}"
          f"{(approx - exact) / exact:>8.2%}")

column                     exact      s    approx      s    error
-----------------------------------------------------------------


GLOBALEVENTID            731,884   0.59   750,507   0.18   2.54%


Actor1Geo_FullName        21,534   0.20    23,164   0.17   7.57%


---

## 2. `rsd` is a standard deviation, not a bound

`approx_count_distinct` takes a parameter named `rsd` — *relative standard deviation*. Its default
is 0.05. The chapter says the summary's "size sets the accuracy", and the relationship is the one
HyperLogLog gives: halving the error costs four times the state, because the standard error falls
with the square root of the number of registers.

The sweep below reports the estimate at three settings. Read the **signed** error, not its size.

In [3]:
COL = "Actor1Geo_FullName"
exact = truth[COL]
print(f"exact distinct {COL} = {exact:,}\n")
print(f"{'rsd':>8}{'estimate':>12}{'signed error':>15}{'seconds':>10}")
print("-" * 45)
for rsd in [0.20, 0.05, 0.01]:
    est, secs = timed(F.approx_count_distinct(COL, rsd=rsd))
    print(f"{rsd:>8.2f}{est:>12,}{(est - exact) / exact:>14.2%}{secs:>10.2f}")

exact distinct Actor1Geo_FullName = 21,534

     rsd    estimate   signed error   seconds
---------------------------------------------
    0.20      26,759        24.26%      0.07
    0.05      23,164         7.57%      0.10


    0.01      21,466        -0.32%      0.75


### The range Spark will accept, and what it does outside it

`rsd` is not a free dial. Spark's HyperLogLog++ implementation refuses values outside a band, and
the refusal is an `AnalysisException` at *analysis* time — before any data is read.

In [4]:
for rsd in [0.39, 0.395, 0.01, 0.0001]:
    try:
        spark.range(1000).select(F.approx_count_distinct("id", rsd=rsd)).first()
        print(f"rsd={rsd:<8} accepted")
    except Exception as e:
        # The lesson is the first line; the rest is a JVM stack trace.
        print(f"rsd={rsd:<8} rejected -- {str(e).splitlines()[0][:96]}")

rsd=0.39     accepted
rsd=0.395    rejected -- [FAILED_FUNCTION_CALL] Failed preparing of the function `approx_count_distinct` for call. Please


rsd=0.01     accepted


rsd=0.0001   rejected -- [FAILED_FUNCTION_CALL] Failed preparing of the function `approx_count_distinct` for call. Please


### Why "guaranteed to within 1%" is not `rsd=0.01`

Exercise 5 asks for a distinct count *guaranteed* to within one percent, and then asks why setting
`rsd` to 0.01 is not necessarily the answer. The reason is in the parameter's name. A **relative
standard deviation** describes the spread of the estimator over repeated trials: it says that about
two thirds of estimates land within one `rsd` of the truth, which means about **one third land
outside it**. A guarantee is a different kind of claim, and HyperLogLog does not offer one.

One estimate cannot show this, because for a fixed input the sketch is deterministic — run it twice
and you get the same number. So the cell below manufactures independent trials: the table is split
into twenty disjoint slices by a hash of the event id, each slice's exact and approximate counts are
computed, and the twenty relative errors are collected. At `rsd=0.05`, roughly a third of them
should fall outside ±5%.

In [5]:
RSD, SLICES = 0.05, 20
sliced = events.withColumn("slice", F.abs(F.hash("GLOBALEVENTID")) % SLICES)

per_slice = (sliced.groupBy("slice")
             .agg(F.count_distinct(COL).alias("exact"),
                  F.approx_count_distinct(COL, rsd=RSD).alias("approx"))
             .collect())

errors = sorted((r["approx"] - r["exact"]) / r["exact"] for r in per_slice)
outside = [e for e in errors if abs(e) > RSD]

print(f"{SLICES} independent slices, rsd={RSD}")
print(f"  worst under-estimate {min(errors):+.2%}   worst over-estimate {max(errors):+.2%}")
print(f"  mean error {statistics.mean(errors):+.2%}   "
      f"stdev of errors {statistics.stdev(errors):.2%}  (rsd claims {RSD:.0%})")
print(f"  outside +-{RSD:.0%}: {len(outside)} of {SLICES} "
      f"= {len(outside) / SLICES:.0%}  -- a standard deviation, not a bound")
print(f"\n  errors: {', '.join(f'{e:+.1%}' for e in errors)}")

20 independent slices, rsd=0.05
  worst under-estimate -8.63%   worst over-estimate +7.67%
  mean error +0.83%   stdev of errors 4.59%  (rsd claims 5%)
  outside +-5%: 7 of 20 = 35%  -- a standard deviation, not a bound

  errors: -8.6%, -6.5%, -3.3%, -3.1%, -2.5%, -2.5%, -2.5%, -0.9%, +0.1%, +0.9%, +1.8%, +1.9%, +2.1%, +2.3%, +4.8%, +5.1%, +6.1%, +6.3%, +7.5%, +7.7%


The measured standard deviation of those errors is the useful number: it lands near the `rsd` that
was asked for, which is the parameter behaving exactly as documented — and that is the point.
`rsd=0.01` buys a *typical* error near one percent at four hundred times the register count of
`rsd=0.20`, and it still leaves a tail outside. A question that genuinely requires one percent with
certainty is a question for `count_distinct`, and \Cref{tab:AggregateCost} says what that costs:
state proportional to the number of distinct values.

---

## 3. The property that makes a sketch worth more than a sample

§3.1: *"compatible sketches can often be combined without processing the original records again."*
Exercise 6 asks what happens if per-partition estimates are averaged instead. Both questions have
the same answer, and it is visible in four numbers over the same data.

The question is the one from Exercise 5(a): how many distinct places appear **per day**, and how
many distinct places appear **over the whole period**? The second is not recoverable from the first.

In [6]:
daily = (events.groupBy("SQLDATE")
         .agg(F.count_distinct(COL).alias("exact_day"),
              F.approx_count_distinct(COL, rsd=0.05).alias("approx_day"),
              F.hll_sketch_agg(COL).alias("sketch"))
         .cache())
days = daily.count()

sum_of_days   = daily.agg(F.sum("exact_day")).first()[0]
mean_of_days  = daily.agg(F.avg("approx_day")).first()[0]
merged_sketch = daily.select(F.hll_sketch_estimate(F.hll_union_agg("sketch"))).first()[0]

print(f"the truth, one pass over every row        {exact:>10,}")
print(f"union of the {days} daily sketches           {merged_sketch:>10,}"
      f"   {(merged_sketch - exact) / exact:+7.2%}")
print(f"SUM of the {days} daily exact counts        {sum_of_days:>10,}"
      f"   {(sum_of_days - exact) / exact:+7.2%}")
print(f"MEAN of the {days} daily estimates         {mean_of_days:>10,.0f}"
      f"   {(mean_of_days - exact) / exact:+7.2%}")

the truth, one pass over every row            21,534
union of the 35 daily sketches               21,444    -0.42%
SUM of the 35 daily exact counts            45,664   +112.06%
MEAN of the 35 daily estimates              1,310   -93.92%


The sum double-counts every place that appears on more than one day; the mean answers a different
question entirely. **Only the union of the sketches answers the question that was asked**, and it
does so without revisiting a single row — the daily sketches are all it reads. That is what
"mergeable" means, and it is why a dashboard can keep one sketch per day and still answer *distinct
over any date range* correctly.

In [7]:
# The same table, per day, so the shape of the double counting is visible.
(daily.select("SQLDATE", "exact_day", "approx_day")
      .withColumn("approx_error", F.round((F.col("approx_day") - F.col("exact_day"))
                                          / F.col("exact_day"), 4))
      .orderBy(F.desc("exact_day")).show(8))

+--------+---------+----------+------------+
| SQLDATE|exact_day|approx_day|approx_error|
+--------+---------+----------+------------+
|20240104|     7116|      6860|      -0.036|
|20240105|     6987|      7452|      0.0666|
|20240103|     6911|      7161|      0.0362|
|20240102|     6082|      5892|     -0.0312|
|20240106|     5063|      4911|       -0.03|
|20240107|     4361|      4369|      0.0018|
|20240101|     4276|      4293|       0.004|
|20230104|      433|       432|     -0.0023|
+--------+---------+----------+------------+
only showing top 8 rows


### And a sample cannot be scaled into a distinct count

A sample keeps real rows, so an average or a sum can be scaled back up by the sampling fraction. A
**distinct count cannot**, and the reason is worth stating precisely: a value that occurs many times
in the population is almost certain to survive a 10% sample, while a value that occurs once survives
with probability 0.10. Dividing by the fraction therefore over-corrects for the common values and
under-corrects for the rare ones.

In [8]:
FRACTION = 0.10
sample = events.sample(withReplacement=False, fraction=FRACTION, seed=777).cache()

s_rows    = sample.count()
s_distinct = sample.select(COL).distinct().count()
s_mean    = sample.agg(F.avg("AvgTone")).first()[0]
pop_mean  = events.agg(F.avg("AvgTone")).first()[0]

print(f"{FRACTION:.0%} sample: {s_rows:,} rows of {N:,}  ({s_rows / N:.2%} actual)\n")
print(f"  mean AvgTone     sample {s_mean:8.4f}   population {pop_mean:8.4f}"
      f"   {(s_mean - pop_mean) / abs(pop_mean):+7.2%}   <- a sample estimates this well")
print(f"  distinct {COL[:18]:<18} in sample {s_distinct:>7,}   population {exact:>7,}"
      f"   {(s_distinct - exact) / exact:+7.2%}")
print(f"  ... scaled by 1/{FRACTION:.2f}           {s_distinct / FRACTION:>7,.0f}"
      f"                       {(s_distinct / FRACTION - exact) / exact:+7.2%}"
      f"   <- and cannot estimate this")

10% sample: 72,998 rows of 731,884  (9.97% actual)

  mean AvgTone     sample  -2.5248   population  -2.5408    +0.63%   <- a sample estimates this well
  distinct Actor1Geo_FullName in sample   8,185   population  21,534   -61.99%
  ... scaled by 1/0.10            81,850                       +280.10%   <- and cannot estimate this


---

## 4. The other approximate family, and the error it bounds

`approxQuantile` takes a *relative error* rather than a relative standard deviation, and what it
bounds is different again: the error is on the **rank**, not on the value. Asking for the median with
a relative error of 0.1 returns a value whose true rank lies somewhere in the middle 20% of the
data — which may be a long way from the median in value terms if the distribution is flat there.

`relativeError=0` asks for the exact answer and pays for it.

In [9]:
t = time.perf_counter(); exact_median = events.approxQuantile("AvgTone", [0.5], 0.0)[0]
t_exact = time.perf_counter() - t
print(f"{'relativeError':>15}{'median':>12}{'value error':>14}{'seconds':>10}")
print("-" * 51)
print(f"{'0 (exact)':>15}{exact_median:>12.4f}{'--':>14}{t_exact:>10.2f}")
for rel in [0.1, 0.01, 0.001]:
    t = time.perf_counter(); got = events.approxQuantile("AvgTone", [0.5], rel)[0]
    print(f"{rel:>15}{got:>12.4f}{(got - exact_median) / abs(exact_median):>13.2%}"
          f"{time.perf_counter() - t:>10.2f}")

  relativeError      median   value error   seconds
---------------------------------------------------
      0 (exact)     -2.5974            --      0.61


            0.1     -3.2947      -26.85%      0.33


           0.01     -2.5990       -0.06%      0.27


          0.001     -2.6005       -0.12%      0.26


---

## What this measured

**An approximate distinct count is a memory bargain, not an accuracy one.** The exact count keeps a
set that grows with the column's cardinality; the sketch keeps a fixed number of registers whatever
the data does. That is \Cref{tab:AggregateCost}'s third row, and the price is a wrong answer by a
small and *stated* amount.

**The stated amount is a standard deviation, and roughly a third of answers are worse than it.** The
twenty-slice experiment in section 2 shows the spread of errors matching the requested `rsd` and a
third of the slices landing outside it. So `rsd=0.01` is the wrong answer to "guaranteed within 1%"
for a reason that has nothing to do with the number: a guarantee and a standard deviation are
different claims, and only `count_distinct` makes the first one.

**Mergeability is the property that earns the sketch its place.** Thirty-five daily sketches unioned
give the whole period's distinct count to a fraction of a percent, reading no rows. Summing the same
thirty-five *exact* daily counts is wrong by more than a factor of two, because it double-counts
every place seen on more than one day. This is Exercise 6's answer: the correct distributed procedure
merges summaries, and averaging or summing per-partition estimates is not a merge.

**And it is the property a sample does not have.** The 10% sample in section 3 estimates the mean
tone to within a fraction of a percent, and its distinct count is wrong by an order of magnitude in
whichever direction it is scaled — because a value's chance of surviving a sample depends on how
often it occurs, which is exactly the information a distinct count is asking about.

**The two approximate families bound different things.** `rsd` bounds a relative standard deviation
on a cardinality; `approxQuantile`'s `relativeError` bounds a **rank**. Reporting either without
saying which was requested tells a reader nothing about how far the answer might be from the truth —
which is §3.1's standard for an approximate result being usable at all.